In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import scipy.io as sio

def reward_categorized_psth(spikes_path, laser_times_path, save_folder, laser_type, csv_path=None, qc_data=None, 
                        bin_size=0.005, time_window=(-1, 1.5), type_file='pdf', 
                        laser_delay=0.5, laser_duration=0.5,
                        min_trials=3, min_spikes=5):
    """
    generate PSTH plots with categories based on reward outcomes.
    """
    os.makedirs(save_folder, exist_ok=True)
    spikes = np.load(spikes_path, allow_pickle=True)
    mat_data = sio.loadmat(laser_times_path)
    trial_data_df = None
    
    trial_data_df = csv_path
    
    # load from csv trial data
    if trial_data_df is not None:
        print("Categorizing trials using CSV data...")
        
        reward_right_laser_times = []
        reward_left_laser_times = []
        nonreward_right_laser_times = []
        nonreward_left_laser_times = []
        
        reward_right_control_times = []
        reward_left_control_times = []
        nonreward_right_control_times = []
        nonreward_left_control_times = []
        
        # CHANGE HERE FOR SPLIT TRIAL, nvm
        for _, trial in trial_data_df.iterrows(): 

            try:
                trial_time = trial['TimeStart']
                is_laser = trial['IsLaserTrial'] == 1
                is_right = trial['TrialSide'] == 'Right'
                is_rewarded = trial['RMI'] == 'reward'
                
                if is_laser:
                    if is_right:
                        if is_rewarded:
                            reward_right_laser_times.append(trial_time)
                        else:
                            nonreward_right_laser_times.append(trial_time)
                    else:  # Left trial
                        if is_rewarded:
                            reward_left_laser_times.append(trial_time)
                        else:
                            nonreward_left_laser_times.append(trial_time)
                else:  # Control trial
                    if is_right:
                        if is_rewarded:
                            reward_right_control_times.append(trial_time)
                        else:
                            nonreward_right_control_times.append(trial_time)
                    else:  # Left trial
                        if is_rewarded:
                            reward_left_control_times.append(trial_time)
                        else:
                            nonreward_left_control_times.append(trial_time)
            except Exception as e:
                print(f"Error processing trial: {e}")
                # Continue with the next trial
                continue
        
        reward_right_laser_times = np.array(reward_right_laser_times)
        reward_left_laser_times = np.array(reward_left_laser_times)
        nonreward_right_laser_times = np.array(nonreward_right_laser_times)
        nonreward_left_laser_times = np.array(nonreward_left_laser_times)
        
        reward_right_control_times = np.array(reward_right_control_times)
        reward_left_control_times = np.array(reward_left_control_times)
        nonreward_right_control_times = np.array(nonreward_right_control_times)
        nonreward_left_control_times = np.array(nonreward_left_control_times)
        
        all_right_laser = np.concatenate([reward_right_laser_times, nonreward_right_laser_times])
        all_left_laser = np.concatenate([reward_left_laser_times, nonreward_left_laser_times])
        all_right_control = np.concatenate([reward_right_control_times, nonreward_right_control_times])
        all_left_control = np.concatenate([reward_left_control_times, nonreward_left_control_times])
        
        reward_right_laser = reward_right_laser_times
        reward_left_laser = reward_left_laser_times
        nonreward_right_laser = nonreward_right_laser_times
        nonreward_left_laser = nonreward_left_laser_times
        
        reward_right_control = reward_right_control_times
        reward_left_control = reward_left_control_times
        nonreward_right_control = nonreward_right_control_times
        nonreward_left_control = nonreward_left_control_times
        
        # print trial counts for each category
        print(f"Trial categorization summary:")
        print(f"  Right laser trials: {len(all_right_laser)} total")
        print(f"    Reward: {len(reward_right_laser_times)}")
        print(f"    Non-reward: {len(nonreward_right_laser_times)}")
        print(f"  Left laser trials: {len(all_left_laser)} total")
        print(f"    Reward: {len(reward_left_laser_times)}")
        print(f"    Non-reward: {len(nonreward_left_laser_times)}")
        print(f"  Right control trials: {len(all_right_control)} total")
        print(f"    Reward: {len(reward_right_control_times)}")
        print(f"    Non-reward: {len(nonreward_right_control_times)}")
        print(f"  Left control trials: {len(all_left_control)} total")
        print(f"    Reward: {len(reward_left_control_times)}")
        print(f"    Non-reward: {len(nonreward_left_control_times)}")
    else:
        print("No trial data available - using event-based matching")
        
        if len(right_sounds) == 0 or len(left_sounds) == 0:
            raise ValueError("No sound data available in MAT file and no CSV data provided")

        laser_times_right = []
        sound_times_right_laser = []
        for right_sound in right_sounds:
            closest_idx = np.argmin(np.abs(laser_times - right_sound))
            closest_time = laser_times[closest_idx]
            if np.abs(closest_time - right_sound) < 1:  # 1s tolerance
                laser_times_right.append(closest_time)
                sound_times_right_laser.append(right_sound)
        
        laser_times_right = np.array(laser_times_right)
        sound_times_right_laser = np.array(sound_times_right_laser)

        # left sound trials with laser
        laser_times_left = []
        sound_times_left_laser = []
        for left_sound in left_sounds:
            closest_idx = np.argmin(np.abs(laser_times - left_sound))
            closest_time = laser_times[closest_idx]
            if np.abs(closest_time - left_sound) < 1:  # 1s tolerance
                laser_times_left.append(closest_time)
                sound_times_left_laser.append(left_sound)
        
        laser_times_left = np.array(laser_times_left)
        sound_times_left_laser = np.array(sound_times_left_laser)

        # control trials (sounds without laser)
        control_times_left = []
        for left_sound in left_sounds:
            if left_sound not in sound_times_left_laser:
                control_times_left.append(left_sound)
        control_times_left = np.array(control_times_left)

        control_times_right = []
        for right_sound in right_sounds:
            if right_sound not in sound_times_right_laser:
                control_times_right.append(right_sound)
        control_times_right = np.array(control_times_right)
        
        # use these for the 'all' category
        all_right_laser = sound_times_right_laser
        all_left_laser = sound_times_left_laser
        all_right_control = control_times_right
        all_left_control = control_times_left
        
        # empty arrays for reward categories - these are the variables used in process_category function
        reward_right_laser = np.array([])
        reward_left_laser = np.array([])
        nonreward_right_laser = np.array([])
        nonreward_left_laser = np.array([])
        
        reward_right_control = np.array([])
        reward_left_control = np.array([])
        nonreward_right_control = np.array([])
        nonreward_left_control = np.array([])
        
        # print trial counts
        print(f"Trial counts:")
        print(f"  Right with laser: {len(all_right_laser)}")
        print(f"  Left with laser: {len(all_left_laser)}")
        print(f"  Right control: {len(all_right_control)}")
        print(f"  Left control: {len(all_left_control)}")
        print("  (No reward/nonreward categorization available)")

    # Load laser times and trial times from MAT file (needed even with CSV)
    try:
        laser_times = mat_data[laser_type][0, 0]['Ts'].flatten()
        print(f"Successfully loaded {len(laser_times)} laser timestamps")
        
        # try to load right and left sounds
        if 'right_sounds_evt07' in mat_data and 'left_sounds_evt08' in mat_data:
            try:
                right_sounds = mat_data['right_sounds_evt07'][0, 0]['Ts'].flatten()
                left_sounds = mat_data['left_sounds_evt08'][0, 0]['Ts'].flatten()
                print(f"Successfully loaded {len(right_sounds)} right sounds and {len(left_sounds)} left sounds")
            except (KeyError, IndexError, AttributeError) as e:
                print(f"Error accessing sound data fields: {e}")
                right_sounds = np.array([])
                left_sounds = np.array([])
        else:
            print("Warning: 'right_sounds_evt07' or 'left_sounds_evt08' not found in MAT file")
            right_sounds = np.array([])
            left_sounds = np.array([])
    except (KeyError, IndexError, AttributeError) as e:
        print(f"Error loading laser times: {e}")
        print("Available fields in MAT file:")
        for key in mat_data.keys():
            if not key.startswith('__'):
                print(f"  - {key}")
        raise ValueError(f"Could not load required data from MAT file: {e}")
        
    if len(laser_times) == 0:
        raise ValueError("No laser timestamps found in MAT file")
    
    if len(right_sounds) == 0 and len(left_sounds) == 0 and trial_data_df is not None:
        print("No sound data found in MAT file, will rely solely on CSV data for categorization")

    # unit IDs to analyze
    if qc_data is not None:
        qc = pd.read_csv(qc_data)
        units = qc.iloc[:, 0].values
    else:
        units = np.unique(spikes['unit_index'])

    # align spikes to sound times
    def align_spikes_to_sound(sound_times, neuron_spikes, time_window):
        if len(sound_times) == 0:
            return [], 0
            
        aligned_spikes = []
        spikes_count = 0
        for st in sound_times:
            spike_window = neuron_spikes[(neuron_spikes >= st + time_window[0]) & 
                                         (neuron_spikes <= st + time_window[1])]
            
            spikes_count += len(spike_window)
            spike_window_aligned = spike_window - st
            aligned_spikes.append(spike_window_aligned)
        return aligned_spikes, spikes_count

    # PSTH function 
    def compute_psth(aligned_spikes, bin_size, time_window):
        bin_edges = np.arange(time_window[0], time_window[1] + bin_size, bin_size)
        num_bins = len(bin_edges) - 1
        num_trials = len(aligned_spikes)
        
        if num_trials == 0:
            # return empty arrays if no trials
            return bin_edges, np.zeros(num_bins), np.zeros(num_bins), 0
            
        spike_counts = np.zeros((num_trials, num_bins))
        
        for i, trial_spikes in enumerate(aligned_spikes):
            counts, _ = np.histogram(trial_spikes, bins=bin_edges)
            spike_counts[i, :] = counts
            
        # convert counts to firing rates (spikes per second)
        firing_rates = spike_counts / bin_size
        
        # compute mean and standard error across trials
        mean_rates = np.mean(firing_rates, axis=0)
        std_rates = np.std(firing_rates, axis=0) / np.sqrt(num_trials) if num_trials > 1 else np.zeros(num_bins)
        
        return bin_edges, mean_rates, std_rates, num_trials

    # calcuate area under curve during laser period
    def calculate_laser_area(bin_centers, mean_rates, laser_onset=0.5, laser_duration=0.5):
        laser_start_idx = np.searchsorted(bin_centers, laser_onset)
        laser_end_idx = np.searchsorted(bin_centers, laser_onset + laser_duration)
        
        if laser_start_idx < len(mean_rates) and laser_end_idx <= len(mean_rates):
            return np.trapz(mean_rates[laser_start_idx:laser_end_idx], 
                           bin_centers[laser_start_idx:laser_end_idx])
        return 0

    def plot_psth_comparison(bin_edges, laser_data, control_data, title, save_path,
                            laser_onset=0.5, laser_duration=0.5):
        plt.figure(figsize=(12, 8))
        bin_centers = bin_edges[:-1] + np.diff(bin_edges) / 2
        
        # laser
        laser_mean, laser_std, laser_trials = laser_data
        lower_bound_laser = np.maximum(laser_mean - laser_std, 0)
        upper_bound_laser = laser_mean + laser_std
        plt.plot(bin_centers, laser_mean, 'r-', label='Laser trials', linewidth=2)
        plt.fill_between(bin_centers, lower_bound_laser, upper_bound_laser, 
                         color='r', alpha=0.2)
        
        # control condition
        control_mean, control_std, control_trials = control_data
        lower_bound_control = np.maximum(control_mean - control_std, 0)
        upper_bound_control = control_mean + control_std
        plt.plot(bin_centers, control_mean, 'k-', label='Control trials', linewidth=2)
        plt.fill_between(bin_centers, lower_bound_control, upper_bound_control, 
                 color='k', alpha=0.2)
        
        # highlight laser period
        plt.axvspan(laser_onset, laser_onset + laser_duration, 
                   color='blue', alpha=0.2, label='Laser period')
        
        # add sound marker at 0
        plt.axvline(0, color='green', linestyle='--', linewidth=1.5, 
                   label='Sound onset (t=0)')
        
        # calculate and display area difference
        laser_area = calculate_laser_area(bin_centers, laser_mean, laser_onset, laser_duration)
        control_area = calculate_laser_area(bin_centers, control_mean, laser_onset, laser_duration)
        area_diff = laser_area - control_area
        area_percent = (area_diff / control_area * 100) if control_area > 0 else 0
        
        plt.text(0.02, 0.95, 
                f'Area during laser period:\nLaser: {laser_area:.2f}\nControl: {control_area:.2f}\nDiff: {area_diff:.2f} ({area_percent:.1f}%)',
                transform=plt.gca().transAxes, fontsize=10,
                bbox=dict(facecolor='white', alpha=0.8))
        
        # add trial count to the plot
        plt.text(0.02, 0.80, 
                f'Trial counts:\nLaser: {laser_trials}\nControl: {control_trials}',
                transform=plt.gca().transAxes, fontsize=10,
                bbox=dict(facecolor='white', alpha=0.8))
        
        plt.xlabel('Time from sound onset (s)')
        plt.ylabel('Firing Rate (Hz)')
        plt.title(title)
        plt.legend(loc='best')
        plt.ylim(bottom=0)
        plt.grid(alpha=0.3)
        
        # save
        plt.savefig(save_path, format=type_file, dpi=300)
        plt.close()

    # create folders for different conditions
    all_dir = os.path.join(save_folder, 'all')
    reward_dir = os.path.join(save_folder, 'reward')
    nonreward_dir = os.path.join(save_folder, 'nonreward')
    
    for directory in [all_dir, reward_dir, nonreward_dir]:
        os.makedirs(directory, exist_ok=True)
    
    # track processed units for summary
    processed_units = {
        'all': {'right': [], 'left': []},
        'reward': {'right': [], 'left': []},
        'nonreward': {'right': [], 'left': []}
    }
    
    # process each unit
    for unit in units:
        print(f"Processing unit {unit}...")
        
        # get neuron spikes
        neuron_spikes = spikes['sample_index'][spikes['unit_index'] == unit]
        neuron_spikes = neuron_spikes / 40000  # Convert spike times to seconds
        
        # helper function to process each category
        def process_category(category_name, right_laser, left_laser, right_control, left_control, output_dir, 
                             enforce_thresholds=True):
            # process right sounds
            aligned_spikes_right_laser, sc_right_laser = align_spikes_to_sound(
                right_laser, neuron_spikes, time_window)
            bin_edges, mean_rates_right_laser, std_rates_right_laser, n_trials_right_laser = compute_psth(
                aligned_spikes_right_laser, bin_size, time_window)
            
            aligned_spikes_right_control, sc_right_control = align_spikes_to_sound(
                right_control, neuron_spikes, time_window)
            _, mean_rates_right_control, std_rates_right_control, n_trials_right_control = compute_psth(
                aligned_spikes_right_control, bin_size, time_window)
            
            # process left sounds
            aligned_spikes_left_laser, sc_left_laser = align_spikes_to_sound(
                left_laser, neuron_spikes, time_window)
            _, mean_rates_left_laser, std_rates_left_laser, n_trials_left_laser = compute_psth(
                aligned_spikes_left_laser, bin_size, time_window)
            
            aligned_spikes_left_control, sc_left_control = align_spikes_to_sound(
                left_control, neuron_spikes, time_window)
            _, mean_rates_left_control, std_rates_left_control, n_trials_left_control = compute_psth(
                aligned_spikes_left_control, bin_size, time_window)
            
            # threshold for right sounds
            has_enough_right = (n_trials_right_laser >= min_trials and 
                               n_trials_right_control >= min_trials and
                               sc_right_laser >= min_spikes and 
                               sc_right_control >= min_spikes)
            
            # threshold for left sounds
            has_enough_left = (n_trials_left_laser >= min_trials and 
                              n_trials_left_control >= min_trials and
                              sc_left_laser >= min_spikes and 
                              sc_left_control >= min_spikes)
            
            # plot right sounds 
            if has_enough_right or not enforce_thresholds:
                if n_trials_right_laser > 0 and n_trials_right_control > 0:  # At least have some trials
                    plot_psth_comparison(
                        bin_edges, 
                        (mean_rates_right_laser, std_rates_right_laser, n_trials_right_laser),
                        (mean_rates_right_control, std_rates_right_control, n_trials_right_control),
                        f"Unit {unit} - Right Sound PSTH ({category_name})",
                        os.path.join(output_dir, f"unit_{unit}_right.{type_file}"),
                        laser_onset=laser_delay, 
                        laser_duration=laser_duration
                    )
                    processed_units[category_name]['right'].append(unit)
                    print(f"Unit {unit}: Generated {category_name} right sound PSTH")
                    right_succeeded = True
                else:
                    print(f"Unit {unit}: No trials available for {category_name} right sounds")
                    right_succeeded = False
            else:
                print(f"Unit {unit}: Skipping {category_name} right sounds - insufficient data")
                print(f"  Right laser: {sc_right_laser} spikes, {n_trials_right_laser} trials")
                print(f"  Right control: {sc_right_control} spikes, {n_trials_right_control} trials")
                right_succeeded = False
            
            if has_enough_left or not enforce_thresholds:
                if n_trials_left_laser > 0 and n_trials_left_control > 0: 
                    plot_psth_comparison(
                        bin_edges, 
                        (mean_rates_left_laser, std_rates_left_laser, n_trials_left_laser),
                        (mean_rates_left_control, std_rates_left_control, n_trials_left_control),
                        f"Unit {unit} - Left Sound PSTH ({category_name})",
                        os.path.join(output_dir, f"unit_{unit}_left.{type_file}"),
                        laser_onset=laser_delay, 
                        laser_duration=laser_duration
                    )
                    processed_units[category_name]['left'].append(unit)
                    print(f"Unit {unit}: Generated {category_name} left sound PSTH")
                    left_succeeded = True
                else:
                    print(f"Unit {unit}: No trials available for {category_name} left sounds")
                    left_succeeded = False
            else:
                print(f"Unit {unit}: Skipping {category_name} left sounds - insufficient data")
                print(f"  Left laser: {sc_left_laser} spikes, {n_trials_left_laser} trials")
                print(f"  Left control: {sc_left_control} spikes, {n_trials_left_control} trials")
                left_succeeded = False
                
            return right_succeeded, left_succeeded, has_enough_right, has_enough_left
        
        all_right, all_left, has_enough_right_all, has_enough_left_all = process_category(
            "all", 
            all_right_laser, all_left_laser, 
            all_right_control, all_left_control,
            all_dir,
            enforce_thresholds=True
        )
        
        if len(reward_right_laser) > 0 or len(reward_left_laser) > 0:
            reward_right, reward_left, _, _ = process_category(
                "reward", 
                reward_right_laser, reward_left_laser, 
                reward_right_control, reward_left_control,
                reward_dir,
                enforce_thresholds=not (has_enough_right_all or has_enough_left_all)
            )
        
        if len(nonreward_right_laser) > 0 or len(nonreward_left_laser) > 0:
            nonreward_right, nonreward_left, _, _ = process_category(
                "nonreward", 
                nonreward_right_laser, nonreward_left_laser, 
                nonreward_right_control, nonreward_left_control,
                nonreward_dir,
                enforce_thresholds=not (has_enough_right_all or has_enough_left_all)
            )
    
    # print summary of processed units
    print("\nProcessing Summary:")
    print(f"Total units: {len(units)}")
    print(f"All category - Right sounds: {len(processed_units['all']['right'])} units")
    print(f"All category - Left sounds: {len(processed_units['all']['left'])} units")
    print(f"Reward category - Right sounds: {len(processed_units['reward']['right'])} units")
    print(f"Reward category - Left sounds: {len(processed_units['reward']['left'])} units")
    print(f"Nonreward category - Right sounds: {len(processed_units['nonreward']['right'])} units")
    print(f"Nonreward category - Left sounds: {len(processed_units['nonreward']['left'])} units")
    

In [3]:

# Check MATLAB behavior file for split delay and start and end blocks

########################## VARIABLES TO CHANGE ###########################
session_id = "Rec_Upstream_SNr_4_250616_500ms_5mW_061625001"

# path set for PLEXON COMPUTER, DCN
base_folder = fr"E:\Paolo\temp_recordings\Upstream_SNr_4\{session_id}"

bin_size = 0.02
laser_onset_1 = 0.0
laser_duration = 0.5 # 500ms
output_folder_psth_1 = "laser_psth_20bin_100reward_500delay"





# if split trial, above variables used for first block
split_trial = False
#split_trial = True
block_1_end = 171 # end of block 1
block_2_start = 190
block_2_end = 390 # end of block 2
laser_onset_2 = 0.0 # 500ms
output_folder_psth_2 = "laser_psth_20bin_500reward_0delay"

####################### DO NOT CHANGE BELOW ########################

output_folder_psth_1_path = rf"{base_folder}\spikeinterface\{output_folder_psth_1}"
output_folder_psth_2_path = rf"{base_folder}\spikeinterface\{output_folder_psth_2}"
spikes_path = rf"{base_folder}\spikeinterface\analyzer\sorting\spikes.npy"
laser_timestamps_path = rf"{base_folder}\{session_id}.mat"
reward_trials_path = rf"{base_folder}\{session_id}_analysis.mat"
laser_type = "laser_on_evt05"
csv_path_csv = rf"{base_folder}\trial_data.csv"
full_trial_data_df = pd.read_csv(csv_path_csv)

trials_block1_df = full_trial_data_df.iloc[:block_1_end].copy()
trials_block2_df = full_trial_data_df.iloc[block_2_start:block_2_end].copy()

csv_path = full_trial_data_df

# if not split trial
if not split_trial:
    reward_categorized_psth(spikes_path=spikes_path,laser_times_path=laser_timestamps_path,
                            save_folder=output_folder_psth_1_path, csv_path=csv_path,
                            laser_type=laser_type,bin_size=bin_size,laser_delay=laser_onset_1,
                            laser_duration=laser_duration)

# if split trial
if split_trial:
    # first block
    reward_categorized_psth(spikes_path=spikes_path,laser_times_path=laser_timestamps_path,
                            save_folder=output_folder_psth_1_path, csv_path=trials_block1_df,
                            laser_type=laser_type,bin_size=bin_size,laser_delay=laser_onset_1,
                            laser_duration=laser_duration)
    # second block
    reward_categorized_psth(spikes_path=spikes_path,laser_times_path=laser_timestamps_path,
                            save_folder=output_folder_psth_2_path, csv_path=trials_block2_df,
                            laser_type=laser_type,bin_size=bin_size,laser_delay=laser_onset_2,
                            laser_duration=laser_duration)

#evt_laser = 'evt_timestamps'

Categorizing trials using CSV data...
Trial categorization summary:
  Right laser trials: 76 total
    Reward: 72
    Non-reward: 4
  Left laser trials: 58 total
    Reward: 57
    Non-reward: 1
  Right control trials: 181 total
    Reward: 177
    Non-reward: 4
  Left control trials: 181 total
    Reward: 180
    Non-reward: 1
Successfully loaded 139 laser timestamps
Successfully loaded 258 right sounds and 244 left sounds
Processing unit 0...
Unit 0: Generated all right sound PSTH
Unit 0: Generated all left sound PSTH
Unit 0: Generated reward right sound PSTH
Unit 0: Generated reward left sound PSTH
Unit 0: Generated nonreward right sound PSTH
Unit 0: Generated nonreward left sound PSTH
Processing unit 1...
Unit 1: Generated all right sound PSTH
Unit 1: Generated all left sound PSTH
Unit 1: Generated reward right sound PSTH
Unit 1: Generated reward left sound PSTH
Unit 1: Generated nonreward right sound PSTH
Unit 1: Generated nonreward left sound PSTH
Processing unit 2...
Unit 2: Gen